# Exploratory Data Analysis (EDA) – Superstore Retail Sales

**Objective:** Analyze customer purchase volume, monthly seasonality, category-wise revenue, regional/segment performance, and discount sensitivity.

**Dataset:** `superstore_sales.csv`  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn

> Run the notebook from top to bottom. Keep `superstore_sales.csv` in the same folder as this notebook.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Load the dataset
file_path = Path("superstore_sales.csv")
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

## 2. Initial data inspection

In [ ]:
print("Shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\\nMissing values:")
missing = df.isna().sum().to_frame("Missing Values")
missing["Missing %"] = (missing["Missing Values"] / len(df) * 100).round(2)
display(missing.sort_values("Missing Values", ascending=False))

print("\\nExact duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 3. Data cleaning and validation

The supplied file contains 10,800 rows, but only 9,994 rows have valid order dates and numeric sales/profit values. The remaining 806 rows are incomplete/non-transaction records (including return-related rows). They are excluded from sales analysis.

The code below parses dates, converts numeric fields, removes exact duplicates, and retains valid sales transactions.

In [ ]:
# Work on a copy
data = df.copy()

# Parse dates and numeric columns
data["Order Date"] = pd.to_datetime(data["Order Date"], errors="coerce")
data["Ship Date"] = pd.to_datetime(data["Ship Date"], errors="coerce")

numeric_cols = ["Row ID", "Postal Code", "Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Remove exact duplicate records
data = data.drop_duplicates()

# Keep complete transaction rows needed for this analysis
required_cols = ["Order ID", "Order Date", "Customer ID", "Category",
                 "Sales", "Quantity", "Discount", "Profit"]
sales = data.dropna(subset=required_cols).copy()

# Additional date features
sales["Year"] = sales["Order Date"].dt.year
sales["Month"] = sales["Order Date"].dt.month
sales["Month Name"] = sales["Order Date"].dt.strftime("%b")
sales["Year-Month"] = sales["Order Date"].dt.to_period("M").astype(str)

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(sales))
print("Rows excluded:", len(df) - len(sales))
print("Cleaned dataset shape:", sales.shape)
display(sales.head())

## 4. Key performance indicators (KPIs)

In [ ]:
total_sales = sales["Sales"].sum()
total_profit = sales["Profit"].sum()
total_orders = sales["Order ID"].nunique()
total_customers = sales["Customer ID"].nunique()
total_quantity = sales["Quantity"].sum()
profit_margin = total_profit / total_sales * 100

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Unique Orders",
               "Unique Customers", "Units Sold", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_orders,
              total_customers, total_quantity, profit_margin]
})
display(kpis)

## 5. Customer purchase volume and segment analysis

In [ ]:
customer_orders = (sales.groupby(["Customer ID", "Customer Name"])
                    .agg(Orders=("Order ID", "nunique"),
                         Sales=("Sales", "sum"),
                         Profit=("Profit", "sum"),
                         Units=("Quantity", "sum"))
                    .sort_values("Sales", ascending=False))

print("Top 10 customers by sales:")
display(customer_orders.head(10))

segment_summary = sales.groupby("Segment").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Customers=("Customer ID", "nunique")
).sort_values("Sales", ascending=False)
display(segment_summary)

segment_summary["Sales"].plot(kind="bar", figsize=(8, 4), title="Sales by Customer Segment")
plt.ylabel("Sales ($)")
plt.xlabel("Customer Segment")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Monthly sales seasonality

In [ ]:
monthly_sales = sales.groupby("Year-Month").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_sales, x="Year-Month", y="Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Average sales by calendar month across all years
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
seasonality = sales.groupby("Month Name")["Sales"].sum().reindex(month_order)
seasonality.plot(kind="bar", figsize=(10, 4), title="Total Sales by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(monthly_sales.sort_values("Sales", ascending=False).head(10))

## 7. Category-wise and sub-category revenue/profit

In [ ]:
category_summary = sales.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
).sort_values("Sales", ascending=False)
category_summary["Profit Margin (%)"] = category_summary["Profit"] / category_summary["Sales"] * 100
display(category_summary)

category_summary["Sales"].plot(kind="bar", figsize=(8, 4), title="Revenue by Product Category")
plt.ylabel("Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

subcategory_summary = sales.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Sales", ascending=False)
display(subcategory_summary)

plt.figure(figsize=(12, 6))
sns.barplot(data=subcategory_summary.reset_index(), x="Sales", y="Sub-Category", hue="Category", dodge=False)
plt.title("Sales by Sub-Category")
plt.xlabel("Sales ($)")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.show()

## 8. Regional performance

In [ ]:
region_summary = sales.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)
region_summary["Profit Margin (%)"] = region_summary["Profit"] / region_summary["Sales"] * 100
display(region_summary)

region_summary[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 5), title="Sales and Profit by Region")
plt.ylabel("Amount ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Discount sensitivity and profitability

This section compares profit across discount levels. It is descriptive: discount levels may be associated with other factors (product, category, quantity, and customer mix), so the results do not prove that discount alone caused a profit change.

In [ ]:
discount_summary = sales.groupby("Discount").agg(
    Transactions=("Discount", "size"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean")
).reset_index()
discount_summary["Profit Margin (%)"] = discount_summary["Profit"] / discount_summary["Sales"] * 100
display(discount_summary)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=sales, x="Discount", y="Profit", alpha=0.35)
plt.axhline(0, linestyle="--")
plt.title("Discount vs. Profit")
plt.xlabel("Discount")
plt.ylabel("Profit ($)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(data=sales, x="Discount", y="Profit")
plt.axhline(0, linestyle="--")
plt.title("Profit Distribution by Discount Level")
plt.xlabel("Discount")
plt.ylabel("Profit ($)")
plt.tight_layout()
plt.show()

## 10. Executive business findings

In [ ]:
# Generate evidence-backed findings from the cleaned data
best_category = category_summary["Sales"].idxmax()
best_category_sales = category_summary.loc[best_category, "Sales"]
best_profit_category = category_summary["Profit"].idxmax()

best_region = region_summary["Sales"].idxmax()
best_region_sales = region_summary.loc[best_region, "Sales"]

worst_discount = discount_summary.loc[discount_summary["Average_Profit"].idxmin()]
negative_discount_levels = discount_summary.loc[discount_summary["Profit"] < 0, "Discount"].tolist()

peak_month = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
best_calendar_month = seasonality.idxmax()

print(f"1. The dataset contains {len(sales):,} valid sales transactions/lines after excluding {len(df)-len(sales):,} incomplete or non-sales rows.")
print(f"2. Total sales were ${total_sales:,.2f}, with total profit of ${total_profit:,.2f} and a profit margin of {profit_margin:.2f}%.")
print(f"3. {best_category} generated the highest category revenue (${best_category_sales:,.2f}); {best_profit_category} generated the highest total profit.")
print(f"4. {best_region} was the highest-revenue region (${best_region_sales:,.2f}).")
print(f"5. {best_calendar_month} had the highest cumulative sales across calendar months in the four-year period.")
print(f"6. The highest-sales individual month was {peak_month['Year-Month']} (${peak_month['Sales']:,.2f}).")
print(f"7. Discount level {worst_discount['Discount']:.0%} had the lowest average profit per line (${worst_discount['Average_Profit']:,.2f}).")
print(f"8. Discount levels with negative aggregate profit: {negative_discount_levels}. Review these discount bands before applying broad promotions.")

## 11. Conclusion and recommendations

- Prioritize profitable categories and sub-categories, not revenue alone.
- Review high-discount transactions where aggregate profit is negative.
- Plan inventory and marketing around the observed monthly sales peaks.
- Compare regional sales and profit margins when allocating sales resources.
- Use customer-level purchase history to identify high-value and repeat customers.

**Caution:** This is a historical, descriptive analysis. Validate findings against business context before making pricing or inventory decisions.